# Phase 1: Forward BCE Math & Numerical Stability

In this section, we will implement the **Binary Cross-Entropy (BCE)** loss function. BCE is used for binary classification because it heavily penalizes predictions that are confidently wrong.

## 1. The Intuition Behind BCE

The mathematical formula for calculating BCE over $N$ examples is:
$$E = -\frac{1}{N} \sum_{n=1}^{N} [y_n \log(p_n) + (1 - y_n) \log(1 - p_n)]$$

- $y_n$ is the true label (0 for Benign, 1 for Malignant).
- $p_n$ is our predicted probability that the tumor is Malignant.

### Why the negative sign?
Probabilities are always between 0 and 1. The logarithm of any value between 0 and 1 is a negative number! Since our goal is to minimize "Loss" (and we want a positive number to represent positive "cost"), we add a negative sign in front to flip the result back to positive.

### How it works
Remember that for any single example, $y_n$ can only be 0 or 1.
- If $y_n = 1$, the right half of the sum `(1 - 1)*...` disappears. The formula becomes just $- \log(p_n)$. If we predict $p_n$ close to 1, loss approaches 0. If we predict close to 0, the loss skyrockets.
- If $y_n = 0$, the left half disappears. The formula becomes just $- \log(1 - p_n)$. Similar logic applies.

* To undestand this deeply **visit the following documentation**: https://app.notion.com/p/jvalenci/Binary-Cross-entropy-Loss-3749d52658e080458d1be1b193770618

In [1]:
import numpy as np

def naive_bce(y_true, y_pred):
    """
    Calculates BCE directly from the formula without protections.
    """
    # y_true.shape[0] gives us the number of samples in the batch ( the rows in the input data )
    N = y_true.shape[0]
    return - (1 / N) * np.sum(y_true * np.log(y_pred) + (1 - y_true) * np.log(1 - y_pred))

In [2]:

# Let's test this with a confidently wrong prediction.
# Example target: 1 (Malignant), our prediction is 0.0 (Extremely confident it's Benign)
y_t = np.array([1])
y_p = np.array([0.0])

# This will cause a numpy RuntimeWarning: divide by zero encountered in log
loss = naive_bce(y_t, y_p)
print(f"Loss: {loss}")

Loss: inf


Loss: inf


/var/folders/lt/0wgz0hyd67z6zxh62hn6qff40000gn/T/ipykernel_8307/316868457.py:9: RuntimeWarning: divide by zero encountered in log
  return - (1 / N) * np.sum(y_true * np.log(y_pred) + (1 - y_true) * np.log(1 - y_pred))


The naive version above outputs `inf` (infinity) and causes a `RuntimeWarning: divide by zero encountered in log` because `log(0)` is undefined. This happens if our network confidently outputs exactly **0.0** or **1.0**.

### Numerical Stability (The Fix)
To avoid infinite values and crashes, we use `np.clip` to enforce that predictions never mathematically hit 0.0 or 1.0. We clip them to a very small range like `[1e-15, 1 - 1e-15]`.

In [3]:
def stable_bce(y_true, y_pred):
    """
    Calculates BCE with numerical stability logic to handle log(0) issues.
    """
    # Clip y_pred between epsilon and 1-epsilon
    epsilon = 1e-15
    y_pred_clipped = np.clip(y_pred, epsilon, 1 - epsilon)
    
    N = y_true.shape[0]
    return - (1 / N) * np.sum(y_true * np.log(y_pred_clipped) + (1 - y_true) * np.log(1 - y_pred_clipped))


In [4]:

# Let's test the same "dangerous" scenario again:
stable_loss = stable_bce(y_t, y_p)
print(f"Successfully calculated stable loss: {stable_loss:.5f}")

Successfully calculated stable loss: 34.53878
